### tests/test_jacobian.ipynb
- tests/test_jacobian.py — 두 구현의 교차 검증 (14강)

In [ ]:
%run -i ../src/transforms.ipynb
%run -i ../src/fk_chain.ipynb
%run -i ../src/jacobian.ipynb

In [ ]:
import numpy as np
import pytest

# tests/test_jacobian.py — 두 구현의 교차 검증
@pytest.mark.parametrize("thetas", [(0.1, 0.5), (1.2, -0.7), (0.0, 1.57)])
def test_analytic_matches_numeric(thetas):
    arm = PlanarArm([0.3, 0.2])
    Ja = jacobian_planar(arm.L, thetas)
    Jn = jacobian_numeric(arm.fk, thetas)
    np.testing.assert_allclose(Ja, Jn, atol=1e-6)

In [ ]:
def test_singularity_detected_when_arm_straight():
    # 팔을 쭉 편 자세(theta2 = 0)는 2링크 평면 팔의 경계 특이점
    J = jacobian_planar([0.3, 0.2], (0.4, 0.0))
    m = singularity_metrics(J)
    assert m["manipulability"] < 1e-6   # sqrt(det) 이라 수치 잔차가 커 보임
    assert m["sigma_min"] < 1e-12

In [ ]:
# 노트북 안에서 테스트 실행 (pytest.mark.parametrize 케이스까지 전개)
_passed = 0
for _name, _fn in list(globals().items()):
    if not (_name.startswith("test_") and callable(_fn)):
        continue
    _ps = [m for m in getattr(_fn, "pytestmark", []) if m.name == "parametrize"]
    if _ps:
        _argnames = [a.strip() for a in _ps[0].args[0].split(",")]
        for _vals in _ps[0].args[1]:
            _fn(*(_vals if len(_argnames) > 1 else (_vals,)))
            _passed += 1
    else:
        _fn(); _passed += 1
    print("PASS", _name)
print(f"{_passed} passed")